# Week 2 – Lasso, Ridge, and Elastic Net Regression

## Objective

This notebook applies regularized linear regression methods to the NBA game total prediction project. The goal is to compare ordinary linear regression with Ridge, Lasso, and Elastic Net regression and evaluate whether regularization improves predictive performance or model interpretability.

The target variable is `actual_total`, representing the total points scored in an NBA game.

Predictors:
- Sportsbook total
- Point spread
- Home moneyline
- Away moneyline

Models are evaluated using R², MAE, and RMSE.

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    root_mean_squared_error
)
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)


In [2]:
merged = pd.read_csv("../data/processed/merged_games_betting.csv")

/var/folders/fj/fkly19253s58fj8xwtwph5cw0000gn/T/ipykernel_94530/89616676.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv("../data/processed/merged_games_betting.csv")


In [3]:
features = [
    "total",
    "spread",
    "moneyline_home",
    "moneyline_away"
]

target = "actual_total"
model_df = merged[features + [target]].dropna()

X = model_df[features]
y = model_df[target]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

## Feature Scaling

Ridge, Lasso, and Elastic Net apply penalties to model coefficients. Because these penalties depend on coefficient magnitude, the predictor variables are standardized before modeling so that variables measured on different scales are treated comparably. The scaler is fit only on the training data to avoid data leakage.

In [5]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
def evaluate_model(name, y_true, y_pred):
    return {
        "Model": name,
        "R²": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred)
    }

results = []

## Baseline Linear Regression

In [7]:
linear = LinearRegression()

linear.fit(X_train_scaled, y_train)

linear_pred = linear.predict(X_test_scaled)

results.append(
    evaluate_model(
        "Linear Regression",
        y_test,
        linear_pred
    )
)

## Ridge Regression

In [8]:
ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_train)

ridge_pred = ridge.predict(X_test_scaled)

results.append(
    evaluate_model(
        "Ridge",
        y_test,
        ridge_pred
    )
)

## Lasso Regression

In [9]:
lasso = Lasso(alpha=0.1)

lasso.fit(X_train_scaled, y_train)

lasso_pred = lasso.predict(X_test_scaled)

results.append(
    evaluate_model(
        "Lasso",
        y_test,
        lasso_pred
    )
)

## Elastic Net

In [10]:
elastic = ElasticNet(
    alpha=0.1,
    l1_ratio=0.5,
    random_state=42
)

elastic.fit(X_train_scaled, y_train)

elastic_pred = elastic.predict(X_test_scaled)

results.append(
    evaluate_model(
        "Elastic Net",
        y_test,
        elastic_pred
    )
)

## Results Analysis

In [11]:
results_df = pd.DataFrame(results)

results_df

,Model,R²,MAE,RMSE
0,Linear Regression,0.386082,13.886242,17.582981
1,Ridge,0.386083,13.886239,17.582967
2,Lasso,0.386211,13.882979,17.581138
3,Elastic Net,0.385833,13.897912,17.586549


## Coefficient Comparison

Regularization affects model coefficients differently. Ridge shrinks coefficients toward zero while retaining all predictors. Lasso can shrink coefficients exactly to zero, effectively performing feature selection. Elastic Net combines both Ridge and Lasso penalties.

In [12]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Linear": linear.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_,
    "ElasticNet": elastic.coef_
})

coef_df

,Feature,Linear,Ridge,Lasso,ElasticNet
0,total,13.973448,13.972507,13.881782,13.262289
1,spread,0.215928,0.215941,0.000000,0.125580
2,moneyline_home,0.308999,0.308853,0.026570,0.140564
3,moneyline_away,0.071600,0.071430,-0.000000,-0.008390


## Conclusion
Regularization techniques did not substantially improve predictive accuracy over ordinary least squares regression for this dataset. However, they improved model interpretability by identifying which predictors contributed the most to forecasting NBA game totals. These findings suggest that sportsbook betting totals contain most of the predictive signal available among the variables analyzed.